# Causal IQ-Learn on AntMaze Medium

In [1]:
import random
import copy
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import ContinuousActor
from causal_rl.algo.imitation.gail.causal_gail import *
from causal_rl.algo.imitation.iqlearn.core_net import IQLearnQNetwork
from causal_rl.algo.imitation.iqlearn.causal_iqlearn import (
    IQLearnReplayBuffer, iq_init_expert_buffer,
    rollout_iqlearn_episode, iqlearn_update_critic, iqlearn_update_actor,
    soft_update, evaluate_iqlearn_policy,
)

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 1
hidden_dims = {'O'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = AntMazePCH(env_id='antmaze-medium-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, O hidden
eval_env = AntMazePCH(env_id='antmaze-medium-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = AntMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'A0', 'A1', 'J0', 'J1', 'L0', 'L1', 'P0', 'P1', 'T0', 'T1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 379882 trajectories


In [8]:
dims = {
    'P': 3,
    # 'O': 4,
    'A': 8,
    'L': 3,
    'T': 3,
    'J': 8,
    'W': 2,
    'X': 8,
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window
causal_Z_trim = trim_Z_sets(Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags
causal_encode, causal_z_dim, causal_slots = build_windowed_z_encoder(
    causal_Z_trim,
    dims=dims,
    lookback=lookback,
)

encode = causal_encode
z_dim = causal_z_dim
Z_trim = causal_Z_trim
causal_z_dim

58

## Hyperparameters

In [10]:
# Shared SAC hyperparameters
total_timesteps = 2_000_000
batch_size = 256
gamma = 0.99
tau = 0.005
actor_lr = 3e-4
critic_lr = 3e-4
alpha_lr = 3e-4
hidden_dim = 256
buffer_capacity = 1_000_000
expert_capacity_ratio = 0.5
start_steps = 5_000
log_every = 50
eval_episodes = 10
max_grad_norm = 1.0
max_updates_per_episode = 1000

# Actor architecture (match GAIL)
num_blocks_actor = 3
dropout_actor = 0.05
layernorm_actor = True

# IQ-Learn specific
num_v_samples = 5

# Environment action space
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())
target_entropy = -float(action_dim)

## Network Initialization

In [11]:
actor = ContinuousActor(
    num_inputs=z_dim, num_outputs=action_dim,
    hidden_size=hidden_dim, std=0.0,
    action_low=action_low, action_high=action_high,
    num_blocks=num_blocks_actor, dropout=dropout_actor, layernorm=layernorm_actor,
).to(device)

q1 = IQLearnQNetwork(z_dim, action_dim, hidden_dim).to(device)
q2 = IQLearnQNetwork(z_dim, action_dim, hidden_dim).to(device)
tq1 = copy.deepcopy(q1)
tq2 = copy.deepcopy(q2)
for p in tq1.parameters(): p.requires_grad = False
for p in tq2.parameters(): p.requires_grad = False

actor_optim = torch.optim.Adam(actor.parameters(), lr=actor_lr)
q1_optim = torch.optim.Adam(q1.parameters(), lr=critic_lr)
q2_optim = torch.optim.Adam(q2.parameters(), lr=critic_lr)

# Automatic entropy tuning
log_alpha = torch.zeros(1, requires_grad=True, device=device)
alpha_optim = torch.optim.Adam([log_alpha], lr=alpha_lr)

buffer = IQLearnReplayBuffer(buffer_capacity, expert_capacity_ratio)
iq_init_expert_buffer(records, encode, buffer, device)

Expert buffer: 379882 transitions from 1000 episodes


## Training

In [12]:
best_eval = -float('inf')
best_state_dict = copy.deepcopy(actor.state_dict())

ts = 0
ep = 0
logs = []

while ts < total_timesteps:
    ep_data = rollout_iqlearn_episode(
        train_env, actor, buffer, encode,
        num_steps, device, deterministic=False, seed=seed + 20000 + ep
    )
    ts += ep_data['episode_length']
    ep += 1

    if ts > start_steps and len(buffer.policy_buffer) >= batch_size // 2:
        n_updates = min(ep_data['episode_length'], max_updates_per_episode)
        for _ in range(n_updates):
            alpha_val = log_alpha.exp().item()
            iqlearn_update_critic(
                q1, q2, tq1, tq2, actor, alpha_val, buffer,
                batch_size, gamma, q1_optim, q2_optim,
                device, num_v_samples, max_grad_norm,
            )
            iqlearn_update_actor(
                actor, q1, q2, log_alpha, target_entropy,
                actor_optim, alpha_optim,
                buffer, batch_size, device, max_grad_norm,
            )
            soft_update(q1, tq1, tau)
            soft_update(q2, tq2, tau)

            # Alpha clamping (IQ-Learn stability fix)
            with torch.no_grad():
                log_alpha.clamp_(min=np.log(0.001), max=np.log(0.1))

    if ep % log_every == 0:
        eval_ret = evaluate_iqlearn_policy(
            train_env, actor, encode, num_steps, device, eval_episodes, seed=42
        )
        logs.append({
            'episode': ep, 'timesteps': ts,
            'eval_return': eval_ret, 'train_return': ep_data['episode_return'],
            'alpha': log_alpha.exp().item(),
        })
        print(
            f"[Causal IQ-Learn ep {ep}] "
            f"ts={ts}, eval={eval_ret:.2f}, "
            f"train={ep_data['episode_return']:.2f}, "
            f"alpha={log_alpha.exp().item():.4f}"
        )

        # Best checkpoint tracking
        if eval_ret > best_eval:
            best_eval = eval_ret
            best_state_dict = copy.deepcopy(actor.state_dict())

# Restore best
actor.load_state_dict(best_state_dict)
print(f"Restored best checkpoint with eval={best_eval:.2f}")

[Causal IQ-Learn ep 50] ts=50000, eval=-322.22, train=-242.39, alpha=0.0464


[Causal IQ-Learn ep 100] ts=97355, eval=-265.56, train=-287.00, alpha=0.0467


[Causal IQ-Learn ep 150] ts=140285, eval=-227.22, train=-177.51, alpha=0.0483


[Causal IQ-Learn ep 200] ts=184551, eval=-165.63, train=-92.65, alpha=0.0472


[Causal IQ-Learn ep 250] ts=223590, eval=-263.86, train=-105.08, alpha=0.0494


...

[Causal IQ-Learn ep 2750] ts=1879258, eval=-328.78, train=-196.94, alpha=0.0662


[Causal IQ-Learn ep 2800] ts=1927450, eval=-316.28, train=-450.33, alpha=0.0666


[Causal IQ-Learn ep 2850] ts=1970081, eval=-197.08, train=-262.99, alpha=0.0647


[Causal IQ-Learn ep 2900] ts=1999741, eval=-141.45, train=-266.26, alpha=0.0639


Restored best checkpoint with eval=-64.86


## Evaluation

In [13]:
causal_iqlearn_policy = make_gail_policy(actor, encode, device=device, deterministic=True)
causal_iqlearn_policies = make_shared_policy_dict(causal_iqlearn_policy)

In [14]:
num_eval_eps = 10
causal_iqlearn_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_iqlearn_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(causal_iqlearn_returns)

Starting episode 1/10...


  Episode 1 ended at step 232 (terminated: True, truncated: False).
Starting episode 2/10...


  Episode 2 ended at step 449 (terminated: True, truncated: False).
Starting episode 3/10...


  Episode 3 ended at step 675 (terminated: True, truncated: False).
Starting episode 4/10...


  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/10...


...

  Episode 7 ended at step 287 (terminated: True, truncated: False).
Starting episode 8/10...


  Episode 8 ended at step 505 (terminated: True, truncated: False).
Starting episode 9/10...


  Episode 9 ended at step 431 (terminated: True, truncated: False).
Starting episode 10/10...


  Episode 10 ended at step 1000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


5955

In [15]:
causal_iqlearn_episode_rewards = defaultdict(float)
for rec in causal_iqlearn_returns:
    ep = rec['episode']
    causal_iqlearn_episode_rewards[ep] += float(rec['reward'])

causal_iqlearn_rewards = [causal_iqlearn_episode_rewards[e] for e in range(num_eval_eps)]
sum(causal_iqlearn_rewards) / num_eval_eps

-165.06351228781995

In [16]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'ciqlearn_antmed.pt')

ckpt = {
    'state_dict': actor.state_dict(),
    'z_dim': causal_z_dim,
    'action_dim': action_dim,
    'hidden_size_actor': hidden_dim,
    'num_blocks_actor': num_blocks_actor,
    'dropout_actor': dropout_actor,
    'layernorm_actor': layernorm_actor,
    'final_tanh': True,
    'action_bounds_low': eval_env.env.action_space.low,
    'action_bounds_high': eval_env.env.action_space.high,
    'Z_sets': causal_Z_trim,
    'dims': dims,
    'lookback': lookback,
}

torch.save(ckpt, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/ciqlearn_antmed.pt
